# Live MSCoRe construction judgments
Select a GPU runtime. Run cells 1–5 in order. Three tasks provide a pilot. The 20-task selection includes the same three. Each question is presented in its original Chinese to three personas using one local model. The board records revisions, challenges and votes. This profile stops on deadlock. Exact-match grading uses the released yes/no labels after discussion.

Source: [MSCoRe by 032564yn](https://huggingface.co/datasets/032564yn/MSCoRe), Construction/simple judgments. This fixed prefix is a subset evaluation.

In [ ]:
#@title 1. Install the pinned project revision
readiness = None
import importlib
import importlib.metadata as metadata
import os
import re
import subprocess
import sys
from tempfile import TemporaryDirectory
from pathlib import Path

PROJECT_REVISION = "ccb8afedd95d2bee040d39368e785b76fbbf68ca" #@param {type:"string"}
CHECKOUT_ROOT = Path("/content/intelligible-blackboard-checkouts")
LEGACY_SOURCE = Path("/content/intelligible-blackboard/src").resolve()
REPO_URL = "https://github.com/nikhil0744/intelligible-blackboard-architecture.git"
if not re.fullmatch(r"[0-9a-f]{40}", PROJECT_REVISION):
    raise ValueError("Use an immutable 40-character project commit, not a branch name.")
PROJECT_DIR = CHECKOUT_ROOT / PROJECT_REVISION
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=True)
if not PROJECT_DIR.exists():
    with TemporaryDirectory(prefix="checkout-", dir=CHECKOUT_ROOT) as staging:
        checkout = Path(staging) / "project"
        subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", PROJECT_REVISION], check=True)
        checkout.rename(PROJECT_DIR)
actual_revision = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != PROJECT_REVISION:
    raise ValueError(f"The checkout at {PROJECT_DIR} has unexpected revision {actual_revision}. Use an intact checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR) + "[colab]"], check=True)
os.chdir(PROJECT_DIR)
for name in ("numpy", "pandas", "pydantic"):
    loaded = sys.modules.get(name)
    if loaded is not None and getattr(loaded, "__version__", None) != metadata.version(name):
        raise RuntimeError("A loaded package changed during installation. Restart the Colab session, then rerun the notebook.")
# Swap only source modules belonging to earlier notebook checkouts.
source_dir = (PROJECT_DIR / "src").resolve()
old_sources = {LEGACY_SOURCE}
old_sources.update(path.resolve() / "src" for path in CHECKOUT_ROOT.iterdir()
                   if path != PROJECT_DIR and re.fullmatch(r"[0-9a-f]{40}", path.name))
def from_previous_checkout(path):
    return bool(path) and any(Path(path).resolve().is_relative_to(old) for old in old_sources)

stale = [name for name, module in tuple(sys.modules.items())
         if from_previous_checkout(getattr(module, "__file__", None))]
if stale and "broker" in globals():
    broker.close()
for name in stale:
    del sys.modules[name]
sys.path[:] = [str(source_dir)] + [path for path in sys.path
                                  if path != str(source_dir) and not from_previous_checkout(path)]
importlib.invalidate_caches()
from benchmarks import batch as benchmark_batch
if not Path(benchmark_batch.__file__).resolve().is_relative_to(source_dir):
    raise RuntimeError("This session loaded a different benchmark package. Restart the Colab session, then rerun cell 1.")

print("Project revision:", actual_revision)
print("Python:", sys.version.split()[0])
print("Benchmark imports ready. Continue to cell 2.")

In [ ]:
#@title 2. Select tasks and mount Drive
from google.colab import drive
TASK_COUNT = 3 #@param [3, 20] {type:"raw"}
if TASK_COUNT not in (3, 20):
    raise ValueError("Select 3 or 20 tasks.")
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/blackboard_benchmarks/mscore")
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 3. Verify the dataset and start local GPU inference
import hashlib
import json
import urllib.request
from benchmarks.batch import check_readiness
from benchmarks.colab_setup import ensure_service
from benchmarks.preparation import resources
from llm_broker.config import InferenceSettings, build_broker

ready = None
url = "https://huggingface.co/datasets/032564yn/MSCoRe/resolve/796e5e0/Data2/Construction/simple/data.json"
digest = "d1d787c2b9fcc0db6d8f0db373e7bd938114fb86a500170951b4f22684c9d87c"
cache = ROOT / "construction-simple.json"
if cache.exists() and hashlib.sha256(cache.read_bytes()).hexdigest() == digest:
    raw = cache.read_bytes()
else:
    with urllib.request.urlopen(url, timeout=90) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != digest:
        raise ValueError("MSCoRe download failed checksum verification.")
    cache.write_bytes(raw)
rows = json.loads(raw)
settings = InferenceSettings(model=resources()["runtime"]["model"], ollama_host="http://127.0.0.1:11434")
service = ensure_service(cpu_smoke=False)
if "broker" in globals():
    broker.close()
broker = build_broker(settings)
ready = check_readiness(broker, settings, cpu_smoke=False)
print("Ready", settings.model, ready["model_digest"])

In [ ]:
#@title 4. Run live discussions and save each result
from datetime import datetime, timezone
from uuid import uuid4
from agents.board_client import InMemoryBoard
from benchmarks.actions import BenchmarkAgent, active_root
from benchmarks.runtime import BenchmarkScheduler, ROSTER
from contracts.schemas import BlackboardStatus
from llm_broker.ledger import usage_scope
from prompts.budget import build_prompt
from scheduler.policies import RoundRobinPolicy

class JudgmentAgent(BenchmarkAgent):
    def output_schema(self):
        schema = super().output_schema()
        revision = schema if self._phase == "submit" else schema["$defs"]["AnswerRevision"]
        revision["properties"]["output"] = {"type": "string", "enum": ["是", "否"]}
        return schema

    def read_with_report(self, snapshot, extra_instructions=None, must_keep_ids=(), *, reserved_tokens=0):
        root = active_root(snapshot)
        instructions = "Assess the Chinese construction statement. Output 是 for true or 否 for false. Explain your judgment."
        if root:
            instructions += f" Independently review the claim. Ratify if supported, otherwise challenge or revise. Current root ID: {root.contribution_id}."
            must_keep_ids = (*must_keep_ids, root.contribution_id)
        else:
            instructions += " Submit an opening revision using kind=final."
        budget = self.prompt_token_budget
        return build_prompt(self.persona, snapshot, self.agent_id, self.system_addendum, self.max_history,
                            token_budget=None if budget is None else max(0, budget-reserved_tokens),
                            must_keep_ids=must_keep_ids, output_contract=instructions + " " + (extra_instructions or ""),
                            next_action="Return the required JSON action.", empty_board="(no contributions yet)")

def discuss(task_id, question):
    sid = str(uuid4())
    board = InMemoryBoard()
    board.create_session(task_id, question, {}, sid)
    scheduler = BenchmarkScheduler(sid, board, policy=RoundRobinPolicy(), consensus_threshold=3,
                                   deadlock_window=2, answer_type="string_exact")
    for name, persona in ROSTER:
        agent = JudgmentAgent(name, persona, broker, model=settings.model, seed=0,
                              counterfactual_capable=False, answer_type="string_exact", **settings.agent_kwargs())
        def turn(snapshot, scheduled, agent=agent):
            return agent.action(snapshot, phase="review_closed" if active_root(snapshot) else "submit")
        scheduler.register_agent(name, agent.role, turn)
    status, error = "turn_limit", None
    with usage_scope(trial_id=sid, session_id=sid):
        try:
            for _ in range(15):
                post = scheduler.step()
                if post is None:
                    status = "queue_empty"
                    break
                print(f"  [{post.agent_id}] {post.tag.value} {post.payload.prediction.claim}", flush=True)
                state = board.get_snapshot(sid).status
                if state in (BlackboardStatus.CONSENSUS, BlackboardStatus.DEADLOCK):
                    status = state.value.lower()
                    break
        except Exception as exc:
            status, error = "error", f"{type(exc).__name__}: {exc}"
    snapshot = board.get_snapshot(sid)
    root = active_root(snapshot)
    return dict(task_id=task_id, question=question, status=status, error=error,
                answer=root.payload.prediction.claim if root and root.metadata.get("final_submission") else None,
                votes=scheduler.vote_state(snapshot), trace=[p.model_dump(mode="json") for p in snapshot.contributions],
                usage=broker.trial_usage(sid).model_dump())

def save(path, record):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

if ready is None:
    raise RuntimeError("Complete cell 3 first.")
selected = rows[:TASK_COUNT]
assert len(selected) == TASK_COUNT and all(row["answer"] in ("是", "否") for row in selected)
run = ROOT / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8])
run.mkdir(parents=True)
ids = [f"construction-simple-{i:04d}" for i in range(1, TASK_COUNT+1)]
save(run / "manifest.json", dict(task_ids=ids, source=url, sha256=digest, project=PROJECT_REVISION,
                                 settings=settings.to_dict(), readiness=ready, seed=0))
print("SAVE THIS RUN FOLDER:", run, flush=True)
for task_id, row in zip(ids, selected):
    print(task_id, row["question"], flush=True)
    result = discuss(task_id, row["question"])
    result.update(reference=row["answer"], correct=result["answer"] == row["answer"])
    save(run / (task_id + ".json"), result)
    print(result["status"], "answer", result["answer"], "correct", result["correct"], result["error"] or "", flush=True)

In [ ]:
#@title 5. Inspect results and optionally download the traces
import shutil
import pandas as pd
from google.colab import files
DOWNLOAD_ZIP = False #@param {type:"boolean"}
results = [json.loads(p.read_text()) for p in sorted(run.glob("construction-simple-*.json"))]
total = len(json.loads((run / "manifest.json").read_text())["task_ids"])
print("Saved", len(results), "/", total, "| Correct", sum(r["correct"] for r in results),
      "| Consensus", sum(r["status"] == "consensus" for r in results))
display(pd.DataFrame([{k:r[k] for k in ("task_id","answer","reference","correct","status","error")} for r in results]))
if DOWNLOAD_ZIP:
    files.download(shutil.make_archive(str(Path("/content") / run.name), "zip", run))